# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения (до запуска):

1. `20 10`: `s["b"]` выбирает по ярлыку, `s.iloc[0]` — по позиции.
2. `Алматы: NaN`, `Астана: NaN`, `Шымкент: 3.0` (Series, `float64`). Pandas складывает по ярлыкам; только Шымкент присутствует в обеих Series. В стандартной сортировке индекс идёт Алматы, Астана, Шымкент.
3. `(2, 2)`: остаются строки с x = 2 и 3, оба столбца сохраняются.
4. `Series DataFrame`: один столбец через строку — Series, через список — DataFrame.
5. `[1, 2] 5`: фильтр сохраняет исходные индексы; `iloc[0, 1]` берёт первую оставшуюся строку, второй столбец.


In [ ]:
from pathlib import Path
# Поддерживаем имя из LMS и имя скачанного вложения.
candidates = [Path("orders.csv"), Path("orders (3).csv"),
              Path("upload/orders (3).csv"), Path("../upload/orders (3).csv")]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = next((Path(name) for name in uploaded if name.endswith(".csv")), None)
    except ImportError:
        pass
if data_path is None:
    raise FileNotFoundError("Положите orders.csv или orders (3).csv рядом с ноутбуком.")
print("Файл данных:", data_path)


Файл данных: ../upload/orders (3).csv


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(data_path)
# Готовая строка из условия: перевод текстовых сумм в числа.
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)


(5463, 8)


In [ ]:
# Проверяем предсказания из раздела «Чтение кода».
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
example = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(example[example["x"] > 1].shape)
print(type(example["x"]).__name__, type(example[["x"]]).__name__)
f = example[example["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])


20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [ ]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [ ]:
print("Строк:", df.shape[0], "| столбцов:", df.shape[1])
print("Типы после преобразования amount:\n", df.dtypes)
print("Пропусков delivery_days:", df["delivery_days"].isna().sum())
for col in ["city", "category", "status", "customer_id"]:
    print(col, "— разных значений:", df[col].nunique())
print("Менеджеру: 1) amount исходно записан текстом: перед суммой нужен перевод в числа.")
print("2) 14 написаний городов дробят один город на несколько: объединить варианты перед сравнением выручки.")
print("3) TEST содержит 3 тестовых заказа: исключить их из реальной выручки, иначе она завышена.")
print("Дополнительно: 160 пропусков доставки и", len(df)-df["order_id"].nunique(),
      "лишних строк по order_id. Пропуски искажают показатели доставки, повторы — выручку.")


Строк: 5463 | столбцов: 8
Типы после преобразования amount:
 order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Пропусков delivery_days: 160
city — разных значений: 14
category — разных значений: 5
status — разных значений: 2
customer_id — разных значений: 2003
Менеджеру: 1) amount исходно записан текстом: перед суммой нужен перевод в числа.
2) 14 написаний городов дробят один город на несколько: объединить варианты перед сравнением выручки.
3) TEST содержит 3 тестовых заказа: исключить их из реальной выручки, иначе она завышена.
Дополнительно: 160 пропусков доставки и 180 лишних строк по order_id. Пропуски искажают показатели доставки, повторы — выручку.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [ ]:
amount_paid = df.loc[paid & not_test, "amount"]
q1 = amount_paid.quantile(0.25)
q3 = amount_paid.quantile(0.75)
print("Заказов:", amount_paid.count())
print(f"Среднее: {amount_paid.mean():,.2f} ₸; медиана: {amount_paid.median():,.2f} ₸")
print(f"25-й перцентиль: {q1:,.2f} ₸; 75-й: {q3:,.2f} ₸")
print("Менеджеру: типичный оплаченный заказ — около 10 300 ₸; центральные 50% — от 5 880 до 18 690 ₸.")
print("Покажите медиану как типичный чек: дорогие заказы тянут среднее вверх. Среднее полезно добавить отдельно.")


Заказов: 4901
Среднее: 14,672.39 ₸; медиана: 10,300.00 ₸
25-й перцентиль: 5,880.00 ₸; 75-й: 18,690.00 ₸
Менеджеру: типичный оплаченный заказ — около 10 300 ₸; центральные 50% — от 5 880 до 18 690 ₸.
Покажите медиану как типичный чек: дорогие заказы тянут среднее вверх. Среднее полезно добавить отдельно.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [ ]:
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancelled = df["status"] == "отменён"
sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)
for name, mask in [("Все города", df["city"].isin(df["city"].unique())),
                   ("Шымкент", sh_mask), ("Алматы", al_mask)]:
    print(f"{name}: {(cancelled & mask).sum()} из {mask.sum()}, доля {cancelled[mask].mean():.2%}")
print(f"Шымкент / Алматы: {cancelled[sh_mask].mean()/cancelled[al_mask].mean():.2f} раза")
print(f"Шымкент / все города: {cancelled[sh_mask].mean()/cancelled.mean():.2f} раза")
print("Менеджеру: Шымкент выделяется большей долей отмен; проверьте причины отмен и сроки доставки в городе.")


Все города: 559 из 5463, доля 10.23%
Шымкент: 164 из 804, доля 20.40%
Алматы: 172 из 2188, доля 7.86%
Шымкент / Алматы: 2.59 раза
Шымкент / все города: 1.99 раза
Менеджеру: Шымкент выделяется большей долей отмен; проверьте причины отмен и сроки доставки в городе.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [ ]:
changes = {}
for name, city_mask in [("Шымкент", sh_mask), ("Алматы", al_mask)]:
    revenue_sep = df.loc[sep & paid & not_test & city_mask, "amount"].sum()
    revenue_oct = df.loc[octo & paid & not_test & city_mask, "amount"].sum()
    changes[name] = (revenue_oct / revenue_sep - 1) * 100
    print(f"{name}: сентябрь {revenue_sep:,.0f} ₸; октябрь {revenue_oct:,.0f} ₸; изменение {changes[name]:.2f}%")
print(f"Падение в процентах в Шымкенте сильнее в {abs(changes['Шымкент']/changes['Алматы']):.2f} раза.")
print("Менеджеру: подробнее проверить Шымкент. Это предварительное сравнение: повторы заказов пока сохранены по условию.")


Шымкент: сентябрь 4,949,800 ₸; октябрь 4,044,010 ₸; изменение -18.30%
Алматы: сентябрь 15,523,940 ₸; октябрь 14,643,020 ₸; изменение -5.67%
Падение в процентах в Шымкенте сильнее в 3.22 раза.
Менеджеру: подробнее проверить Шымкент. Это предварительное сравнение: повторы заказов пока сохранены по условию.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# Ошибка 1: без скобок & связывается раньше сравнений, возникает ошибка выполнения.
# Ошибка 2: == "Шымкент" пропускает написания "шымкент" и "Shymkent".
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
exact_sh = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("Отменённых заказов Шымкента:", len(cancel_sh))
print("При одном написании:", len(exact_sh), "| пропущено:", len(cancel_sh)-len(exact_sh))
print("Менеджеру: учитывать все написания города, иначе число отмен занижено.")


Отменённых заказов Шымкента: 164
При одном написании: 153 | пропущено: 11
Менеджеру: учитывать все написания города, иначе число отмен занижено.


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
counts = df["customer_id"].value_counts()
top_customer = counts.index[0]
client = df.loc[df["customer_id"] == top_customer]
print("Клиент:", top_customer, "| строк заказов:", counts.iloc[0])
print(df.loc[df["customer_id"] == top_customer,
             ["order_date", "city", "amount", "status"]].to_string(index=False))
id_counts = client["order_id"].value_counts()
print("Повторяющиеся order_id:\n", id_counts[id_counts > 1])
print("Разных заказов клиента:", client["order_id"].nunique())
print("Менеджеру: C01818 имеет 11 строк, но 10 разных order_id. Заказ 101504 на 8 890 ₸ повторён.")
print("Число строк завышает активность клиента и выручку; проверить повторы перед окончательным отчётом.")


Клиент: C01818 | строк заказов: 11
      order_date      city  amount  status
2026-09-01 15:00   Шымкент  7350.0 оплачен
2026-09-09 18:00   Шымкент 45270.0 оплачен
2026-09-15 21:00    Алматы  8890.0 оплачен
2026-09-15 21:00    Алматы  8890.0 оплачен
2026-09-24 18:00    Астана  1490.0 оплачен
2026-09-29 09:00    Астана 22170.0 оплачен
2026-09-30 21:00    Алматы 20180.0 оплачен
2026-10-08 14:00 Караганда  4330.0 оплачен
2026-10-18 08:00    Астана 13600.0 оплачен
2026-10-21 19:00    Алматы 21990.0 оплачен
2026-10-23 18:00    Алматы 11140.0 оплачен
Повторяющиеся order_id:
 order_id
101504    2
Name: count, dtype: int64
Разных заказов клиента: 10
Менеджеру: C01818 имеет 11 строк, но 10 разных order_id. Заказ 101504 на 8 890 ₸ повторён.
Число строк завышает активность клиента и выручку; проверить повторы перед окончательным отчётом.


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
iqr = q3 - q1
limit = q3 + 1.5 * iqr
outliers = df.loc[paid & not_test & (df["amount"] > limit)]
print(f"Q1 = {q1:,.0f} ₸; Q3 = {q3:,.0f} ₸; IQR = {iqr:,.0f} ₸; верхняя граница = {limit:,.0f} ₸")
print(f"Выше границы: {len(outliers)}, доля: {len(outliers)/len(amount_paid):.2%}")
print("Категории:\n", outliers["category"].value_counts())
print("Менеджеру: 320 дорогих заказов (6,53%), чаще электроника — 308. Проверить их, не удалять автоматически.")
print("IQR использует квартильный размах, а z-оценка — среднее и стандартное отклонение.")
print("Дорогие заказы повышают среднее и стандартное отклонение; порог z может оказаться выше.")
print("87 из ДЗ 4 и 320 здесь получены разными правилами; там также использовалась другая подготовка данных (без дублей).")


Q1 = 5,880 ₸; Q3 = 18,690 ₸; IQR = 12,810 ₸; верхняя граница = 37,905 ₸
Выше границы: 320, доля: 6.53%
Категории:
 category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
Менеджеру: 320 дорогих заказов (6,53%), чаще электроника — 308. Проверить их, не удалять автоматически.
IQR использует квартильный размах, а z-оценка — среднее и стандартное отклонение.
Дорогие заказы повышают среднее и стандартное отклонение; порог z может оказаться выше.
87 из ДЗ 4 и 320 здесь получены разными правилами; там также использовалась другая подготовка данных (без дублей).


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# Требуемая доля отмен по каждому написанию — одной строкой:
print((df.loc[cancelled, "city"].value_counts() / df["city"].value_counts()).sort_values(ascending=False))

city_counts = df["city"].value_counts()
cancel_counts = df.loc[cancelled, "city"].value_counts()
rates = cancel_counts / city_counts
# Проверка выравнивания по ярлыкам: обратный порядок знаменателя не меняет ответ.
reordered = cancel_counts / city_counts.iloc[::-1]
print("Проверка ярлыков:", rates.sort_index().equals(reordered.sort_index()))
for city in SH:
    print(f"{city}: {city_counts[city]} заказов, {cancel_counts[city]} отмен, доля {rates[city]:.2%}")
print("Менеджеру: по одному написанию нельзя судить обо всём городе: размеры и состав подвыборок различаются.")
print("Перед анализом убрать краевые пробелы, привести регистр и объединить транслитерацию в единые названия.")


city
Shymkent     0.230769
Шымкент      0.215190
алматы       0.140625
астана       0.130952
Актобе       0.103275
Astana       0.100000
Караганда    0.089219
Астана       0.084829
Алматы       0.079917
Атырау       0.075342
Almaty       0.057971
 Алматы      0.045455
шымкент      0.037037
АЛМАТЫ       0.032258
Name: count, dtype: float64
Проверка ярлыков: True
Шымкент: 711 заказов, 153 отмен, доля 21.52%
шымкент: 54 заказов, 2 отмен, доля 3.70%
Shymkent: 39 заказов, 9 отмен, доля 23.08%
Менеджеру: по одному написанию нельзя судить обо всём городе: размеры и состав подвыборок различаются.
Перед анализом убрать краевые пробелы, привести регистр и объединить транслитерацию в единые названия.
